# V12 Engines — Delivery-Delay Preprocessing & Feature Template
**Owner:** Data & preprocessing · **Status:** template for the event (follow the team docs)

This notebook turns the raw USAID SCMS CSV into a model-ready feature table that matches the **Interface & Contract Spec**. Every decision here is from the **Project Guidelines v2** data audit.

> **Rules note:** this is a *template / plan* for understanding and practice. The final production run happens on LinuxONE during the event. Keep it honest: no fabricated numbers, leakage-safe by construction.

**Confirmed decisions baked in:** label = delivered > scheduled · keep From RDC · shipment-mode blanks → "Unknown" · decision date = PO else PQ (left empty when neither exists, never filled with another date) · prior-rate cutoff = decision date, else an **estimated order date** (scheduled − median planned lead time for the shipment's route) · split by scheduled date Option B (train ≤2013 / test 2014–2015) · weight excluded · prior-late-rate = leakage-safe, 365-day window, smoothing M=20, lane = each of country/vendor/route.

**Changed in v3 (Oct 17, after the baseline run):**
1. New column `prior_global_rate`: the smoothed-toward global late rate in the same 365-day window (self excluded), i.e. the value every key prior is shrunk toward.
2. New features `prior_excess_{country,vendor,fulfillment_route}` = key prior − `prior_global_rate`. The raw priors swing with the overall late rate year to year (0.159, 0.235, 0.079, 0.179 for 2010-13) and lag it by up to a year, so across years a high prior often precedes a *good* year. The excess keeps only "this lane is worse than everyone right now", which is what transfers.
3. All v2 columns are unchanged (checked in §8b); tables are saved as `train_v3` / `test_v3`.

**Changed in v2 (Oct 17):**
1. `decision_date` is no longer conceptually tied to the scheduled date. When unknown it stays empty, flagged by `decision_date_known = 0`.
2. The prior-rate cutoff for those rows is now an estimated order date (new column `prior_cutoff_date`, flag `cutoff_estimated`) instead of the scheduled date, which let priors see outcomes delivered after the real order date.
3. A cutoff can never fall after the scheduled date (guards the few rows where the PO/PQ date is after the scheduled date).
4. The shipment itself is now excluded from the **global** prior as well as from its own key's prior.
5. Added §8b: automated checks for leakage, missing values and split counts.

## 0. Config

In [1]:
import pandas as pd, numpy as np

CSV_PATH       = "SCMS_Delivery_History_Dataset_20150929.csv"  # <-- set your path
WINDOW_DAYS    = 365      # recency window for prior late-rate
SMOOTH_M       = 20.0     # smoothing strength (pseudo-counts)
FALLBACK_PRIOR = 0.115    # overall late rate; used only when no prior outcomes exist
TEST_YEARS     = {2014, 2015}   # Option B: train <=2013, test 2014-2015
ESTIMATE_KEY   = "fulfillment_route"  # group used to estimate missing order dates (median planned lead time)

## 1. Load & parse dates
Two date formats in this file: scheduled/delivered are `%d-%b-%y` (e.g. 10-Jan-15); PO/PQ are `%m/%d/%Y` with placeholder text ("N/A - From RDC", "Pre-PQ Process", "Date Not Captured") that becomes NaT.

In [2]:
df = pd.read_csv(CSV_PATH, encoding="latin-1", low_memory=False)
df.columns = [c.strip().lower() for c in df.columns]

scheduled = pd.to_datetime(df["scheduled delivery date"], errors="coerce", format="%d-%b-%y")
delivered = pd.to_datetime(df["delivered to client date"], errors="coerce", format="%d-%b-%y")
po = pd.to_datetime(df["po sent to vendor date"],  errors="coerce", format="%m/%d/%Y")
pq = pd.to_datetime(df["pq first sent to client date"], errors="coerce", format="%m/%d/%Y")

print("rows:", len(df), "| scheduled NaT:", scheduled.isna().sum(), "| delivered NaT:", delivered.isna().sum())

rows: 10324 | scheduled NaT: 0 | delivered NaT: 0


## 2. Label
`late = delivered after scheduled`. `days_late` is the regressor target (clipped at 0). The order/decision date is **not** part of the label — it is only used for the split and the runway.

In [3]:
days_late  = (delivered - scheduled).dt.days
label_late = (days_late > 0).astype(int)
days_late  = days_late.clip(lower=0)
print("late rate: %.1f%% (%d of %d)" % (label_late.mean()*100, label_late.sum(), len(df)))

late rate: 11.5% (1186 of 10324)


## 3. Decision-date anchor
`decision_date = PO sent to vendor, else PQ first sent` (covers ~81%). This is the date the order was actually placed, so when neither date exists it stays **empty** and `decision_date_known = 0`. It is never filled with the scheduled date: that would claim the order was placed on the day it was due to arrive.

The cutoff used for the prior late-rates is a separate concept and is built in §5b.

In [4]:
decision_date       = po.fillna(pq)          # real order/decision date; NaT when unknown
decision_date_known = decision_date.notna()
print("decision_date known: %.1f%%" % (decision_date_known.mean()*100))

decision_date known: 81.3%


## 4. Categorical & numeric fields
Shipment-mode blanks become **"Unknown"** (NOT imputed to Ocean — the data audit showed blanks are 1.1% late vs Ocean 17.5%, so imputing would mislabel).

In [5]:
out = pd.DataFrame()
out["order_id"]           = (df["id"] if "id" in df.columns else pd.Series(range(len(df)))).astype(str)
out["country"]            = df["country"].astype(str)
out["vendor"]             = df["vendor"].astype(str)
out["fulfillment_route"]  = df["fulfill via"].astype(str)         # Direct Drop | From RDC
out["shipment_mode"]      = df["shipment mode"].fillna("Unknown").replace({"": "Unknown"}).astype(str)
out["product_group"]      = df["product group"].astype(str)
out["managed_by"]         = df["managed by"].astype(str)
out["manufacturing_site"] = df["manufacturing site"].astype(str)
out["quantity"]           = pd.to_numeric(df["line item quantity"], errors="coerce")

## 5. Planned lead time
`planned_lead_time_days = scheduled − decision_date` (known for ~81%). A few rows where the anchor is after the scheduled date (data error) → set to NaN. Weight is **excluded** (not reliably known at order time).

In [6]:
plt_days = (scheduled - decision_date).dt.days
plt_days = plt_days.where(plt_days >= 0)   # drop negative (anchor after scheduled)
out["planned_lead_time_days"] = plt_days
out["decision_date_known"]    = decision_date_known.astype(int)

out["decision_date"]           = decision_date
out["scheduled_delivery_date"] = scheduled
out["label_late"]              = label_late
out["days_late"]               = days_late

## 5b. Prior-rate cutoff (estimated order date where the decision date is unknown)
The prior late-rates may only use shipments delivered **before the order was placed**. Where the decision date is unknown we estimate the order date as `scheduled − median planned lead time` for the shipment's fulfillment route.

- The medians are learned **only from training-period rows (≤2013) with a real decision date**, so the test period never informs the estimate.
- The estimate lives in `prior_cutoff_date` (flag `cutoff_estimated = 1`). It is used only to cut off history. It is **not** a model feature and never replaces `decision_date`.
- A cutoff is capped at the scheduled date, for the few rows whose PO/PQ date falls after it (a data-entry error).
- The previous version used the scheduled date itself, which let these rows' priors include outcomes delivered after the real order date (look-ahead).

In [7]:
train_period   = ~scheduled.dt.year.isin(TEST_YEARS)
known_lead     = out["planned_lead_time_days"].where(train_period & decision_date_known)
route_median   = known_lead.groupby(out[ESTIMATE_KEY]).median()
overall_median = known_lead.median()
est_lead       = out[ESTIMATE_KEY].map(route_median).fillna(overall_median)

estimated_order_date = scheduled - pd.to_timedelta(est_lead.round(), unit="D")
cutoff = decision_date.fillna(estimated_order_date)
cutoff = cutoff.where(cutoff <= scheduled, scheduled)   # never after the scheduled date

out["prior_cutoff_date"] = cutoff
out["cutoff_estimated"]  = (~decision_date_known).astype(int)

print("median planned lead time by route (train, known anchor):", route_median.round(0).to_dict(),
      "| overall:", round(overall_median))
print("cutoffs estimated: %d rows | capped at scheduled: %d rows"
      % (out["cutoff_estimated"].sum(), int((decision_date > scheduled).sum())))

median planned lead time by route (train, known anchor): {'Direct Drop': 91.0, 'From RDC': 162.0} | overall: 122
cutoffs estimated: 1931 rows | capped at scheduled: 4 rows


## 6. Leakage-safe prior late-rate (365-day window + smoothing)
For each shipment, the prior late-rate of a key (country / vendor / route) uses **only shipments that had already been DELIVERED before this shipment's cutoff** (`prior_cutoff_date`: the decision date, else the estimated order date from §5b), within the last 365 days, smoothed toward the same-window global prior. The shipment itself is excluded from both its key's count and the global prior, so its own outcome can never enter its features. The data is non-stationary (late rates swing by year), so the recency window matters — it roughly doubled class separation vs an all-history average.

**v3:** the global window rate is also saved as `prior_global_rate`, and `prior_excess_<key> = prior_late_rate_<key> − prior_global_rate` measures how much worse a lane is than the overall rate in the same window.

In [ ]:
def add_prior(key, window_days=WINDOW_DAYS, M=SMOOTH_M, fb=FALLBACK_PRIOR):
    cut = cutoff.values.astype("datetime64[D]")
    dlv = delivered.values.astype("datetime64[D]")
    lab = label_late.values
    kv  = out[key].values
    N   = len(out); res = np.empty(N); known = np.zeros(N, dtype=int); glob = np.empty(N)
    w   = np.timedelta64(window_days, "D")
    go = np.argsort(dlv); gd = dlv[go]; gl = lab[go]; gc = np.concatenate([[0], np.cumsum(gl)])
    def win(sd, cum, c):
        hi = np.searchsorted(sd, c, "left"); lo = np.searchsorted(sd, c - w, "left")
        return hi - lo, cum[hi] - cum[lo]
    tmp = pd.DataFrame({"k": kv, "cut": cut, "dlv": dlv, "lab": lab})
    for _, g in tmp.groupby("k"):
        idx = g.index.values
        o = np.argsort(g["dlv"].values); kd = g["dlv"].values[o]; kl = g["lab"].values[o]
        kc = np.concatenate([[0], np.cumsum(kl)])
        for i in idx:
            c = cut[i]; sd = dlv[i]; sl = lab[i]
            self_in = (sd < c) and (sd >= c - w)      # this shipment falls inside its own window
            n, s = win(kd, kc, c)
            gn, gs = win(gd, gc, c)
            if self_in:                               # exclude self from key AND global counts
                n -= 1; s -= sl; gn -= 1; gs -= sl
            gp = gs/gn if gn > 0 else fb
            res[i] = (s + M*gp) / (n + M) if (n + M) > 0 else fb
            glob[i] = gp
            known[i] = 1 if n > 0 else 0
    return res, known, glob

for key in ["country", "vendor", "fulfillment_route"]:
    r, kn, gl = add_prior(key)
    out[f"prior_late_rate_{key}"] = r
    out[f"prior_known_{key}"]     = kn
    if "prior_global_rate" in out:   # identical for every key (self excluded from the global count)
        assert np.allclose(out["prior_global_rate"].values, gl)
    out["prior_global_rate"] = gl

# v3: lane-relative priors (remove the year-level swing that all keys share)
for key in ["country", "vendor", "fulfillment_route"]:
    out[f"prior_excess_{key}"] = out[f"prior_late_rate_{key}"] - out["prior_global_rate"]

print("prior by route:", out.groupby("fulfillment_route")["prior_late_rate_fulfillment_route"].mean().round(3).to_dict())

## 7. Time-based split (Option B)
Split on the always-present `scheduled` date: train ≤2013, test 2014–2015 (≈25% test, ~356 late cases — enough for stable PR-AUC / recall).

In [9]:
sy = scheduled.dt.year
out["_split"] = np.where(sy.isin(TEST_YEARS), "test", "train")
print(out["_split"].value_counts().to_dict())

{'train': 7779, 'test': 2545}


## 8. Assemble feature matrix (matches the Interface Spec) & sanity checks

In [ ]:
# decision_date_known stays a feature per Playbook §3. Note: it is 1 for every test row and every live
# order, so it can only learn the 2006-2009 gap; the model owner should compare with/without it.
FEATURES = ["country","vendor","fulfillment_route","shipment_mode","product_group",
            "managed_by","manufacturing_site","quantity","planned_lead_time_days","decision_date_known",
            "prior_late_rate_country","prior_late_rate_vendor","prior_late_rate_fulfillment_route",
            "prior_known_country","prior_known_vendor","prior_known_fulfillment_route"]
# v3 additions (offered alongside, not replacing, the Playbook 16 so baselines can compare):
RELATIVE_FEATURES = ["prior_excess_country","prior_excess_vendor","prior_excess_fulfillment_route"]

train = out[out._split=="train"]; test = out[out._split=="test"]
X_train, y_train = train[FEATURES], train["label_late"]
X_test,  y_test  = test[FEATURES],  test["label_late"]
print("train", X_train.shape, "late", int(y_train.sum()))
print("test ", X_test.shape,  "late", int(y_test.sum()))
print("NaN by column:", out.isna().sum()[lambda s: s>0].to_dict())

# Save the cleaned tables (parquet for the model owner, csv for inspection):
try:
    train.to_parquet("train_v3.parquet"); test.to_parquet("test_v3.parquet")
except ImportError:
    print("pyarrow not installed: skipping parquet")
date_cols = ["decision_date", "prior_cutoff_date", "scheduled_delivery_date"]
for name, d in (("train", train), ("test", test)):
    d.assign(**{c: d[c].dt.strftime("%Y-%m-%d") for c in date_cols}).to_csv(f"{name}_v3.csv", index=False)
print("saved train_v3 / test_v3")

## 8b. Checks (must all pass before handing off)
(a) **No leakage:** for a random sample, recount by brute force the same-key shipments delivered inside `[cutoff − 365 days, cutoff)` excluding the row itself, and confirm they reproduce the stored prior. Also confirm the shipment's own outcome never counts.
(b) **Missing values only where expected:** `planned_lead_time_days` and `decision_date`; `prior_cutoff_date` is always filled and never after the scheduled date; `decision_date` is empty exactly where `decision_date_known = 0`.
(c) **Split counts** match the Playbook: train 7,779 / 830 late · test 2,545 / 356 late.

In [ ]:
rng = np.random.default_rng(0)
cut_d = cutoff.values.astype("datetime64[D]"); dlv_d = delivered.values.astype("datetime64[D]")
lab_v = label_late.values; w = np.timedelta64(WINDOW_DAYS, "D")

def brute_prior(i, key):
    same = out[key].values == out[key].values[i]
    inwin = (dlv_d >= cut_d[i] - w) & (dlv_d < cut_d[i])
    others = np.arange(len(out)) != i
    sel = same & inwin & others
    g = inwin & others
    gp = lab_v[g].mean() if g.sum() > 0 else FALLBACK_PRIOR
    n, s = sel.sum(), lab_v[sel].sum()
    return (s + SMOOTH_M*gp) / (n + SMOOTH_M), int(n > 0)

def inwin_global(i):
    g = (dlv_d >= cut_d[i] - w) & (dlv_d < cut_d[i]) & (np.arange(len(out)) != i)
    return lab_v[g].mean() if g.sum() > 0 else FALLBACK_PRIOR

for key in ["country", "vendor", "fulfillment_route"]:
    for i in rng.choice(len(out), 300, replace=False):
        p, k = brute_prior(i, key)
        assert abs(p - out[f"prior_late_rate_{key}"].iat[i]) < 1e-9, (key, i)
        g = inwin_global(i)
        assert abs(g - out["prior_global_rate"].iat[i]) < 1e-9, ("global", i)
        assert k == out[f"prior_known_{key}"].iat[i], (key, i)
print("(a) leakage check passed: key priors and the global rate reproduce from strictly-earlier deliveries, self excluded")

nan_cols = set(out.isna().sum()[lambda s: s>0].index)
assert nan_cols <= {"planned_lead_time_days", "decision_date"}, nan_cols
for key in ["country", "vendor", "fulfillment_route"]:
    assert np.allclose(out[f"prior_excess_{key}"], out[f"prior_late_rate_{key}"] - out["prior_global_rate"])
assert out["prior_cutoff_date"].notna().all()
assert (out["prior_cutoff_date"] <= out["scheduled_delivery_date"]).all()
assert (out["decision_date"].isna() == (out["decision_date_known"] == 0)).all()
print("(b) missing-value check passed:", sorted(nan_cols))

assert (len(train), int(train.label_late.sum())) == (7779, 830)
assert (len(test),  int(test.label_late.sum()))  == (2545, 356)
print("(c) split counts match: train 7,779 / 830 late · test 2,545 / 356 late")

## 9. Honesty / handoff notes
- **No leakage:** prior features use only outcomes delivered before each row's cutoff, with the row itself excluded from both the key and the global prior; checked automatically in §8b.
- **Decision date is never invented.** When PO and PQ are both missing (~19%, almost all 2006–2009) `decision_date` stays empty with `decision_date_known = 0`. Only the prior-rate cutoff is estimated, in its own column, from the training period's median lead time per route.
- **`decision_date_known` is an era marker.** It is 1 for every test row and for any live order (the order date is the moment of scoring), so as a feature it can only learn the early-years gap (rows without a date are ~0.8% late). Model owner: compare the model with and without it, and with 2010–2013 only.
- **Non-stationarity is real** (late rates swing by year) → the 365-day window is deliberate, and it is also the pitch argument for *retraining on current data* in deployment.
- **planned_lead_time NaN** for ~19% (no decision date) plus a few negative leads is expected — keep it as missing (the baseline needs a median fill + flag inside `build_features`); never drop silently.
- **Next (model owner):** class-weighted Logistic Regression baseline + Snap ML boosted-tree classifier (late prob) and regressor (days late); evaluate with PR-AUC and recall at the operating threshold, not accuracy; report the Error breakdown (Type I / Type II).